# Parser Prototyping — Notebook-to-Module Promotion Path

This notebook demonstrates the **notebook → module promotion path** described in
architecture §18 (JupyterHub responsibilities).

## The promotion path

```
exploratory notebook
  → reviewed Python module or parametrized report
  → tests on known runs
  → pinned environment
  → pipeline job type
  → recorded in processing_environment + reproducibility receipt
```

In this notebook we:
1. Read a raw FET transfer file from `labdata.store`.
2. Prototype the parsing logic interactively.
3. Call the **already-promoted** `labdata.parsers.fet_transfer.parse` and
   `compute_metrics` functions to validate our prototype.
4. Explain when and how to promote a notebook prototype to a versioned module.

**Key rule:** notebooks in `/srv/labdata/notebooks` are read-only to the catalog.
A notebook prototype becomes production only after it passes tests and is
committed as a versioned module in `workers/labdata/parsers/`.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import labdata.catalog as catalog
import labdata.store as store
from labdata.parsers import fet_transfer as parser_mod

## Step 1: Read a raw file from `labdata.store`

`store.raw_files(run_id)` resolves the sample/device from the catalog and returns
a list of file paths under `/srv/labdata/raw/<sample>/<device>/<run_id>/`.

We read the first `.data` file as plain text, just as a pipeline worker would.

In [ ]:
# Pick a published run
df = catalog.list_runs(state='published')
if df.empty:
    raise RuntimeError("No published runs. Run labdata.seed.seed_demo() first.")

run_id = df.iloc[0]['id']
print(f"Using run: {run_id}")

# Resolve raw file paths
raw_paths = store.raw_files(run_id)
print(f"Raw files: {[p.name for p in raw_paths]}")

# Find the primary data file
data_files = [p for p in raw_paths if p.suffix in ('.data', '.csv', '.dat', '.txt')]
if not data_files:
    raise RuntimeError(f"No data file found in raw files: {raw_paths}")

data_path = data_files[0]
print(f"Primary data file: {data_path}")

# Read raw text
raw_text = data_path.read_text()
print(f"\nFirst 200 chars of raw file:")
print(raw_text[:200])

## Step 2: Prototype parsing interactively

Here we prototype the core parsing logic as a notebook cell — the same logic
that `labdata.parsers.fet_transfer.parse` implements as a tested module.

This is how you would start developing a new parser for a new measurement type.

In [ ]:
def prototype_parse(text: str):
    """
    Prototype FET transfer parser — reads V,I columns, skips comment/header lines.
    Returns (voltages, currents) as numpy arrays.

    NOTE: This is a prototype. The production version is
    labdata.parsers.fet_transfer.parse — which includes full validation,
    row-count checking, NaN detection, and warning accumulation.
    """
    voltages = []
    currents = []
    in_header = True

    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue

        # Skip the column header line (non-numeric first token)
        if in_header:
            in_header = False
            parts = re.split(r'[,\s]+', line)
            try:
                float(parts[0])
            except ValueError:
                continue  # it's a header row

        # Parse data row
        parts = re.split(r'[,\s]+', line)
        parts = [p for p in parts if p]
        if len(parts) == 2:
            try:
                voltages.append(float(parts[0]))
                currents.append(float(parts[1]))
            except ValueError:
                pass

    return np.array(voltages), np.array(currents)


proto_V, proto_I = prototype_parse(raw_text)
print(f"Prototype parser: {len(proto_V)} rows")
print(f"V range: [{proto_V.min():.2f}, {proto_V.max():.2f}]")
print(f"I range: [{proto_I.min():.2e}, {proto_I.max():.2e}]")

## Step 3: Validate against the production parser

Call the versioned `labdata.parsers.fet_transfer.parse` function and compare
results. If the prototype and the production parser agree, your prototype logic
is ready for review.

The production parser also runs `compute_metrics` to derive scalar summary
metrics (I_max, I_min, on/off ratio, V_th).

In [ ]:
# Run the versioned production parser
prod_result = parser_mod.parse(raw_text)

print(f"Production parser version: {parser_mod.PARSER_VERSION}")
print(f"Parse ok:     {prod_result.ok}")
print(f"Rows actual:  {prod_result.rows_actual}")
if prod_result.warnings:
    print(f"Warnings:     {prod_result.warnings}")

# Cross-check row count
assert len(proto_V) == prod_result.rows_actual, (
    f"Row count mismatch: prototype={len(proto_V)}, production={prod_result.rows_actual}"
)
print("\nPrototype and production parser agree on row count.")

In [ ]:
# Compute metrics with the production function
if prod_result.ok:
    metrics = parser_mod.compute_metrics(prod_result)
    print("Computed metrics:")
    print(f"  I_max:        {metrics.I_max:.3e} A")
    print(f"  I_min:        {metrics.I_min:.3e} A")
    print(f"  On/off ratio: {metrics.on_off_ratio:.2e}")
    if metrics.V_th is not None:
        print(f"  V_th:         {metrics.V_th:.3f} V")

In [ ]:
# Plot: prototype vs production
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(proto_V, np.abs(proto_I), 'b--', label='prototype', linewidth=2)
ax.semilogy(prod_result.voltage, np.abs(prod_result.current), 'r-', label='production', linewidth=1, alpha=0.7)
ax.set_xlabel('Gate Voltage (V)')
ax.set_ylabel('|Drain Current| (A)')
ax.set_title('Prototype vs Production Parser Output')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 4: How to promote a prototype to a pipeline module

Once a prototype parser is validated in this notebook, the promotion path is:

### 1. Extract the logic to a Python module

Create `workers/labdata/parsers/<measurement_type>.py` with:
- `PARSER_VERSION = "<type>_v1.0"` — a version string for the reproducibility receipt
- `parse(text: str) -> ParseResult` — the validated parse function
- `compute_metrics(result: ParseResult) -> Metrics` — scalar metric extraction

### 2. Write table-driven unit tests

Create `workers/tests/test_<measurement_type>.py` with:
- Known-good file → expected row count, column range
- Malformed file → `ok=False`, expected error message
- Trailing NaN rows → warning recorded

### 3. Run tests on known runs

Test the new parser against real data in `labdata.store` using integration tests
(pattern: `workers/tests/test_pipeline_integration.py`).

### 4. Pin the environment

Update `workers/requirements.txt` if new dependencies were introduced.
The processing environment (Python version, package hashes) is recorded in
`processing_environments` and included in the reproducibility receipt.

### 5. Register the job type

Add a new `job_type` constant (e.g. `parse_<measurement_type>`) and a handler
in `workers/labdata/pipeline.py`. The pipeline A worker will then pick up
runs of this measurement type automatically.

### 6. The receipt records the exact version

```text
published_result.parser_version  = "<type>_v1.0"
published_result.processing_git_sha = "<git SHA at job run time>"
published_result.processing_environment_id = <env record id>
```

This means every published result can be re-run deterministically from the
receipt alone — the hard requirement of this system.

## Summary

| Stage | Location | Immutable? |
|---|---|---|
| Prototype cells | This notebook | No — exploratory |
| Reviewed Python module | `workers/labdata/parsers/` | Yes — versioned |
| Unit tests | `workers/tests/test_*.py` | Yes — versioned |
| Pipeline job type | `workers/labdata/pipeline.py` | Yes — versioned |
| Reproducibility receipt | `published_results` table | Yes — immutable |

**The notebook is a sandbox. The module is truth.**